In [1]:
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null
!pip install gitignore_parser > /dev/null
!pip install lightning > /dev/null

In [3]:
from google.colab import drive
import os
import sys
import yaml
import torch
import importlib
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightning import seed_everything

# 1. Environment Configuration
if not os.path.exists('/content/drive/MyDrive'):
  drive.mount('/content/drive')

project_root = '/content/drive/MyDrive/FundGitHubProject'
eomt_folder = project_root + '/eomt'

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if eomt_folder not in sys.path:
    sys.path.insert(0, eomt_folder) # Insert at the beginning to override pre-installed 'datasets'

from eval.iouEval import iouEval

seed_everything(0, verbose=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

Mounted at /content/drive
Active Device: cuda


In [6]:
import os

source = '/content/drive/MyDrive/FundGitHubProject'
destination = '/ProjectFolder'

if not os.path.exists(destination):
    os.symlink(source, destination)
    print("Shortcut created! Refresh the Files menu on the left to see '/ProjectFolder'.")
else:
    print("Shortcut already exists!")

Shortcut already exists!


In [10]:
!cat /ProjectFolder/eomt/training/lightning_module.py

# ---------------------------------------------------------------
# © 2025 Mobile Perception Systems Lab at TU/e. All rights reserved.
# Licensed under the MIT License.
#
# Portions of this file are adapted from:
# - the torchmetrics library by the PyTorch Lightning team
# - the Mask2Former repository by Facebook, Inc. and its affiliates
# All used under the Apache 2.0 License.
# ---------------------------------------------------------------

import math
from typing import Optional, cast
import lightning
from lightning.fabric.utilities import rank_zero_info
import torch
import torch.nn as nn
from torch.optim import AdamW
from torchmetrics.classification import MulticlassJaccardIndex
from torchmetrics.detection import PanopticQuality, MeanAveragePrecision
from torchmetrics.functional.detection._panoptic_quality_common import (
    _prepocess_inputs,
    _Color,
    _get_color_areas,
    _calculate_iou,
)
import wandb
from PIL import Image
import matplotlib.colors as mcolors
from matplo

In [18]:
import os
import sys
from lightning.pytorch import Trainer

# Ensure eomt is in path so main.py can be imported
sys.path.insert(0, '/ProjectFolder/eomt')
import main as eomt_main

# Suppress PyTorch FX warnings for DINOv3 models
os.environ["TORCH_LOGS"] = "-dynamo"

# We will intercept the model here
loaded_model = None

# Monkey-patch Trainer.fit so it doesn't actually train, but captures the model
original_fit = Trainer.fit
def fake_fit(self, model, **kwargs):
    global loaded_model
    loaded_model = model

Trainer.fit = fake_fit

# Backup and mock sys.argv as if running from command line
sys_argv_backup = sys.argv
sys.argv = ['main.py', 'fit', '--config', '/ProjectFolder/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml']

try:
    # Run the exact entry point authored by the developers
    eomt_main.cli_main()

    print("\n\033[1;32m✅ Model loaded successfully!\033[0m")
    print(f"Lightning Module: {type(loaded_model).__name__}")
    print(f"Core Network: {type(loaded_model.network).__name__}")
    print(f"Backbone: {type(loaded_model.network.encoder).__name__}")
finally:
    # Restore system state
    sys.argv = sys_argv_backup
    Trainer.fit = original_fit


INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0
INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (timm/vit_base_patch14_reg4_dinov2.lvd142m)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
INFO:httpx:HTTP Request: HEAD https://huggingface.co/timm/vit_base_patch14_reg4_dinov2.lvd142m/resolve/main/model.safetensors "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/timm/vit_base_patch14_reg4_dinov2.lvd142m/xet-read-token/3b06466a5548c52b8b98822e1

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

INFO:timm.models._hub:[timm/vit_base_patch14_reg4_dinov2.lvd142m] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO:timm.layers.pos_embed:Resized position embedding: (37, 37) to (64, 64).
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installi

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: s360426 (s360426-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✅ Model loaded successfully!
Lightning Module: OptimizedModule
Core Network: EoMT
Backbone: ViT


In [20]:
import json

notebook_path = '/ProjectFolder/inference.ipynb'

with open(notebook_path, 'r', encoding='utf-8') as f:
    notebook = json.load(f)

print("--- Extracting Model Initialization Logic ---\n")
for i, cell in enumerate(notebook.get('cells', [])):
    if cell.get('cell_type') == 'code':
        source = "".join(cell.get('source', []))
        # Look for the cell that instantiates the model
        if 'model =' in source or 'model_kwargs' in source:
            print(f"Found in Cell {i + 1}:\n")
            # Print line by line to prevent Colab from truncating massive string blocks
            for line in source.splitlines():
                print(line)
            print("\n" + "="*50 + "\n")


--- Extracting Model Initialization Logic ---

Found in Cell 7:

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=data.img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=data.num_classes,
    encoder=encode